In [82]:
# Import packages.
import pandas as pd
import numpy as np
import os



In [83]:
# Load data.

# define file path, separate for organization.
netflix_prize_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/combined_data_1.txt"

# data is separated by "," so we can use pd.read_csv().
rows = []
movie_id = None

with open(netflix_prize_filepath, "r") as file:
    for line in file:
        line = line.strip()

        if line.endswith(":"):
            movie_id = line.replace(":", "")
        
        else:
            customer_id, movie_rating, date = line.split(",")
            rows.append([movie_id, customer_id, movie_rating, date ])
            
netflix_prize_data = pd.DataFrame(
    rows,
    columns = ["movie_id", "customer_id", "movie_rating", "date"]  
)

# Convert data to useable format.
netflix_prize_data["movie_id"] = netflix_prize_data["movie_id"].astype(int)
netflix_prize_data["customer_id"] = netflix_prize_data["customer_id"].astype(int)
netflix_prize_data["movie_rating"] = netflix_prize_data["movie_rating"].astype(float)
netflix_prize_data["date"] = pd.to_datetime(netflix_prize_data["date"])

# Preview first couple rows.
netflix_prize_data.head()

,movie_id,customer_id,movie_rating,date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03


In [84]:
# Feature Engineer.
netflix_prize_data["avg_customer_rating"] = netflix_prize_data.groupby("customer_id")["movie_rating"].transform("mean")
netflix_prize_data["avg_movie_rating"] = netflix_prize_data.groupby("movie_id")["movie_rating"].transform("mean")
netflix_prize_data["avg_date_rating"] = netflix_prize_data.groupby("date")["movie_rating"].transform("mean")
netflix_prize_data["count_of_ratings_per_date"] = netflix_prize_data.groupby("date")["movie_rating"].transform("count")
netflix_prize_data["count_of_ratings_per_movie"] = netflix_prize_data.groupby("movie_id")["movie_rating"].transform("count")

# Preview.
netflix_prize_data.head()

,movie_id,customer_id,movie_rating,date,avg_customer_rating,avg_movie_rating,avg_date_rating,count_of_ratings_per_date,count_of_ratings_per_movie
0,1,1488844,3.0,2005-09-06,3.253308,3.749543,3.674945,45986,547
1,1,822109,5.0,2005-05-13,4.083333,3.749543,3.662106,30699,547
2,1,885013,4.0,2005-10-19,3.873563,3.749543,3.729616,40313,547
3,1,30878,4.0,2005-12-26,3.634304,3.749543,3.685429,11832,547
4,1,823519,3.0,2004-05-03,3.917197,3.749543,3.579395,27993,547


In [85]:
# Load Netflix movie titles.

# define file path, separate for organization.
netflix_titles_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/movie_titles.csv"

netflix_titles_data = pd.read_csv(
    netflix_titles_filepath,
    encoding="latin-1",
    header=None,
    names=["movie_id", "year", "title"],
    sep=",",
    engine="python",
    on_bad_lines="skip")

# Convert data to useable format.
netflix_titles_data["movie_id"] = netflix_titles_data["movie_id"].astype(int)
netflix_titles_data["year"] = pd.to_numeric(
    netflix_titles_data["year"],
    errors="coerce"
)

netflix_titles_data["title"] = netflix_titles_data["title"].str.strip()

# checking duplicate titles returned "False", tells us we need to look further into the matter as movies can have duplicate titles with different release years.
netflix_titles_data["title"].is_unique
netflix_titles_data["movie_id"].is_unique

# Further inspection confirms that duplicate movies are released in separate years.
netflix_title_data[
    netflix_title_data["title"].duplicated(keep=False)
].sort_values("title")


# Preview first couple rows.
netflix_titles_data.head()

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


In [ ]:
# Load extra netflix date.
netflix_data_extra_path = "/Users/calynguyen/Downloads/data/NetFlix.csv"
netflix_data_extra = pd.read_csv(
    netflix_data_extra_path,
    )

# Formatting.
netflix_data_extra["type"] = netflix_data_extra["type"].str.strip().astype("string")
netflix_data_extra["title"] = netflix_data_extra["title"].str.strip().astype("string")
netflix_data_extra["director"] = netflix_data_extra["director"].str.strip().astype("string")
netflix_data_extra["cast"] = netflix_data_extra["cast"].str.strip().astype("string")
netflix_data_extra["country"] = netflix_data_extra["country"].str.strip().astype("string")
netflix_data_extra["date_added"] = pd.to_datetime(netflix_data_extra["date_added"])
netflix_data_extra["release_year"] = netflix_data_extra["release_year"].astype("int")

netflix_data_extra["minutes"] = np.nan
netflix_data_extra["seasons"] = np.nan


# Separate duration column, documentation indicates that the duration column consists of seasons and minutes.
netflix_data_extra.loc[
    netflix_data_extra["type"] == "Movie",
    "minutes"
] = netflix_data_extra["duration"]

netflix_data_extra.loc[
    netflix_data_extra["type"] == "TV Show",
    "seasons"
] = netflix_data_extra["duration"]

netflix_data_extra = netflix_data_extra.drop(columns=["rating", "duration"])





netflix_data_extra.head()

/var/folders/rw/dby4248j285dbb5m8qj9nb_80000gn/T/ipykernel_90987/1194977590.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  netflix_data_extra["date_added"] = pd.to_datetime(netflix_data_extra["date_added"])


,show_id,type,title,director,cast,country,date_added,release_year,genres,description,minutes,seasons
0,s1,TV Show,3%,<NA>,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,2020-08-14,2020,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...,NaN,4.0
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,2017-12-15,2008,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...,143.0,NaN
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,2019-01-05,2016,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...,124.0,NaN
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,2016-03-01,2016,Comedies,New NFL star Thad buys his old teammates' belo...,90.0,NaN
4,s1001,TV Show,Blue Planet II,<NA>,David Attenborough,United Kingdom,2018-12-03,2017,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...,NaN,1.0


In [87]:
# Merge data netflix_title_data & netflix_prize_data

data_sources = {
    "rating_data": netflix_prize_data,
    "title_data": netflix_titles_data
}

for name, data in data_sources.items():
    print(name)
    print("Shape:", data.shape)
    display(data.head())

rating_data
Shape: (24053764, 9)


,movie_id,customer_id,movie_rating,date,avg_customer_rating,avg_movie_rating,avg_date_rating,count_of_ratings_per_date,count_of_ratings_per_movie
0,1,1488844,3.0,2005-09-06,3.253308,3.749543,3.674945,45986,547
1,1,822109,5.0,2005-05-13,4.083333,3.749543,3.662106,30699,547
2,1,885013,4.0,2005-10-19,3.873563,3.749543,3.729616,40313,547
3,1,30878,4.0,2005-12-26,3.634304,3.749543,3.685429,11832,547
4,1,823519,3.0,2004-05-03,3.917197,3.749543,3.579395,27993,547


title_data
Shape: (17434, 3)


,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
